# BudgiBot Brain 

In [1]:
# Insalling pip dependecies
!pip install -r requirements.txt

In [2]:
# Setting up embedding model
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/60/jkpcvndn6j1bchnq17ptwvm80000gn/T/ipykernel_35211/3753005436.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [20]:
# Sample training data (labelled user examples)
training_examples = [
    ("Paid rent for August", "Rent"),
    ("Monthly Netflix subscription", "Subscriptions"),
    ("Groceries from Walmart", "Groceries"),
    ("Electricity bill payment", "Utilities"),
    ("Watched a movie at PVR", "Shopping & Entertainment"),
    ("Renewed car insurance", "Insurance"),
    ("Doctor consultation fee", "Health"),
    ("Metro card recharge", "Transport"),
]

# Convert sample data to documents with category label as metadata
from langchain.docstore.document import Document

docs = [Document(page_content=ex[0], metadata={"category": ex[1]}) for ex in training_examples]

In [4]:
# Splitting of sample data into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
split_docs = text_splitter.split_documents(docs)

In [ ]:
# Create FAISS vector store locally
from langchain.vectorstores import FAISS
import os

db = FAISS.from_documents(split_docs, embedding_model)

# Load or create FAISS vector store
if os.path.exists("faiss_store/index.faiss"):
    db = FAISS.load_local("faiss_store", embeddings=embedding_model)
    print("Using available local vector db")
else:
    db = FAISS.from_documents([], embedding_model)
    print("Creating a new vector db")

/opt/homebrew/Caskroom/miniconda/base/envs/budgibot-backend/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


IndexError: list index out of range

In [ ]:
# Creating a prompt template that is passed to the RAG chain every time as RAG is stateless
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = PromptTemplate.from_template("""
You are an AI assistant that classifies short user inputs (descriptions of expenses) into predefined categories.

Your job is to choose the most appropriate category for the given input. The categories are:
- Rent
- Insurance
- Utilities
- Shopping & Entertainment
- Groceries
- Subscriptions
- Transport
- Health

Use the following context as examples:
{context}

Now classify the user's input below and respond ONLY with the category name.

In case you are not sure of the category, prompt the user to provide more information.

User Input: {question}
""")



In [7]:
# Setup Groq LLM
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

In [ ]:
# Retrieves relevant documents from a vector store based on a user query, injects them into a prompt template, and sends the completed prompt to an LLM to generate a final answer.
from langchain.chains import RetrievalQA

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=False,
    chain_type_kwargs={"prompt": SYSTEM_PROMPT}
)

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets

def classify_input(input_text):
    # 1. Get category from the RAG chain
    category = rag_chain.run(input_text).strip()
    # 2. Create a new Document with input and predicted category
    new_doc = Document(page_content=input_text, metadata={"category": category})
    # 3. Add to the FAISS vector store
    db.add_documents([new_doc])
    # 4. Persist to local disk
    db.save_local("faiss_store")
    return category

def interactive_chat():
    input_box = widgets.Text(
        description='Prompt:',
        placeholder='e.g. Bought shoes from Nike',
        layout=widgets.Layout(width='90%')
    )
    output_area = widgets.Output()

    def on_enter(_):
        user_input = input_box.value.strip()
        if user_input:
            category = classify_input(user_input)
            with output_area:
                print(f"Input: {user_input}\n→ Predicted Category: {category}\n")
            input_box.value = ''  # Clear input after submission

    # Attach Enter key event
    input_box.on_submit(on_enter)

    display(input_box, output_area)


In [17]:
interactive_chat

<function __main__.interactive_chat()>